# BCI Focus Session Analysis

This notebook analyses **BCI (Brain-Computer Interface) focus session data** from the Zylvex Mind Mapper app.

**What you'll see:**
- Time-series focus scores with a rolling average overlay
- Focus distribution histogram across three sessions
- Peak focus detection marked on the timeline
- Correlation heatmap between focus score, rolling average, node creation events, and focus derivative
- 3D surface: focus score over time × session number (multi-session comparison)
- An interactive **Sandbox** section to adjust parameters and re-render charts live

All 500-point session data is generated with NumPy — no external files required.

In [ ]:
!pip install numpy plotly scipy ipywidgets --quiet

In [ ]:
import warnings
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import find_peaks
import ipywidgets as widgets
from IPython.display import display

warnings.filterwarnings('ignore')
rng = np.random.default_rng(42)

In [ ]:
# Generate sample BCI session data — 3 sessions × 500 data points each
N = 500

def generate_session(seed):
    rng_s = np.random.default_rng(seed)
    t = np.linspace(0, 10 * 60, N)  # 10-minute session in seconds
    # Random walk base
    walk = np.cumsum(rng_s.normal(0, 0.008, N))
    # Add sine wave variation (engagement cycles)
    sine = 0.12 * np.sin(2 * np.pi * t / 120) + 0.06 * np.sin(2 * np.pi * t / 45)
    focus = 0.55 + walk + sine
    focus = np.clip(focus, 0.0, 1.0)
    # Node creation events: 50 random moments per session
    event_idx = rng_s.choice(N, size=50, replace=False)
    events = np.zeros(N)
    events[event_idx] = 1
    return t, focus, events

sessions = [generate_session(seed) for seed in [42, 99, 7]]

# Convenience aliases for session 1
t1, focus1, events1 = sessions[0]
t2, focus2, events2 = sessions[1]
t3, focus3, events3 = sessions[2]

print(f'Session 1 — mean focus: {focus1.mean():.3f}, max: {focus1.max():.3f}, min: {focus1.min():.3f}')
print(f'Session 2 — mean focus: {focus2.mean():.3f}')
print(f'Session 3 — mean focus: {focus3.mean():.3f}')
print(f'Node creation events in session 1: {int(events1.sum())}')

In [ ]:
# Time-series plot — session 1 with rolling average overlay
def rolling_avg(arr, window):
    return np.convolve(arr, np.ones(window) / window, mode='same')

WINDOW = 30
roll1 = rolling_avg(focus1, WINDOW)

fig_ts = go.Figure()
fig_ts.add_trace(go.Scatter(
    x=t1, y=focus1,
    mode='lines',
    line=dict(color='rgba(99,102,241,0.5)', width=1.5),
    name='Focus Score',
))
fig_ts.add_trace(go.Scatter(
    x=t1, y=roll1,
    mode='lines',
    line=dict(color='#10b981', width=2.5),
    name=f'Rolling Avg ({WINDOW}pt)',
))
fig_ts.update_layout(
    title='BCI Focus Score — Session 1',
    xaxis_title='Time (seconds)',
    yaxis_title='Focus Score',
    yaxis=dict(range=[0, 1]),
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
    yaxis_gridcolor='rgba(255,255,255,0.07)',
    legend=dict(bgcolor='rgba(255,255,255,0.05)', bordercolor='rgba(255,255,255,0.1)', borderwidth=1),
)
fig_ts.show()

In [ ]:
# Focus distribution histogram — all 3 sessions
all_focus = np.concatenate([focus1, focus2, focus3])

fig_hist = go.Figure()
for i, (sess_focus, sess_label, color) in enumerate(
    zip([focus1, focus2, focus3], ['Session 1', 'Session 2', 'Session 3'],
        ['rgba(99,102,241,0.7)', 'rgba(16,185,129,0.7)', 'rgba(245,158,11,0.7)'])
):
    fig_hist.add_trace(go.Histogram(
        x=sess_focus, nbinsx=30, name=sess_label,
        marker_color=color, opacity=0.8,
    ))
fig_hist.update_layout(
    title='Focus Score Distribution — All Sessions',
    barmode='overlay',
    xaxis_title='Focus Score',
    yaxis_title='Count',
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
    yaxis_gridcolor='rgba(255,255,255,0.07)',
    legend=dict(bgcolor='rgba(255,255,255,0.05)', bordercolor='rgba(255,255,255,0.1)', borderwidth=1),
)
fig_hist.show()

In [ ]:
# Peak focus detection — scipy find_peaks
peaks, _ = find_peaks(focus1, height=0.75, distance=20)
print(f'Detected {len(peaks)} focus peaks in session 1')

fig_peaks = go.Figure()
fig_peaks.add_trace(go.Scatter(
    x=t1, y=focus1,
    mode='lines', line=dict(color='rgba(99,102,241,0.6)', width=1.5),
    name='Focus Score',
))
fig_peaks.add_trace(go.Scatter(
    x=t1[peaks], y=focus1[peaks],
    mode='markers',
    marker=dict(symbol='triangle-up', size=12, color='#ef4444',
                line=dict(color='white', width=1)),
    name='Peak Focus',
))
# Threshold line
fig_peaks.add_hline(y=0.75, line_dash='dash', line_color='rgba(245,158,11,0.5)',
                    annotation_text='Peak threshold (0.75)')
fig_peaks.update_layout(
    title='Peak Focus Detection — Session 1',
    xaxis_title='Time (seconds)',
    yaxis_title='Focus Score',
    yaxis=dict(range=[0, 1]),
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
    yaxis_gridcolor='rgba(255,255,255,0.07)',
    legend=dict(bgcolor='rgba(255,255,255,0.05)', bordercolor='rgba(255,255,255,0.1)', borderwidth=1),
)
fig_peaks.show()

In [ ]:
# Correlation heatmap
roll1_full = rolling_avg(focus1, 30)
focus_deriv = np.diff(focus1, prepend=focus1[0])

corr_matrix = np.corrcoef(np.stack([
    focus1, roll1_full, events1, focus_deriv
]))
labels = ['Focus Score', 'Rolling Avg', 'Node Events', 'Focus Δ']

fig_heatmap = go.Figure(go.Heatmap(
    z=corr_matrix,
    x=labels, y=labels,
    colorscale='RdBu',
    zmid=0, zmin=-1, zmax=1,
    text=np.round(corr_matrix, 2),
    texttemplate='%{text}',
    colorbar=dict(title='Correlation'),
))
fig_heatmap.update_layout(
    title='Correlation Heatmap — Session 1 Features',
    paper_bgcolor='#0a0a0f',
    plot_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
)
fig_heatmap.show()

In [ ]:
# 3D surface: focus score over time × session number
all_sessions_focus = np.array([focus1, focus2, focus3])  # shape (3, 500)
time_idx = np.arange(N)
session_idx = np.array([1, 2, 3])

fig_surf = go.Figure(go.Surface(
    x=time_idx,
    y=session_idx,
    z=all_sessions_focus,
    colorscale=[
        [0.0, '#ef4444'],
        [0.5, '#f59e0b'],
        [1.0, '#10b981'],
    ],
    cmin=0, cmax=1,
    opacity=0.9,
    colorbar=dict(title='Focus Score'),
))
fig_surf.update_layout(
    title='Focus Score Surface — Time × Session',
    scene=dict(
        xaxis_title='Time Index',
        yaxis_title='Session',
        zaxis_title='Focus Score',
        zaxis=dict(range=[0, 1]),
        bgcolor='#0a0a0f',
        xaxis=dict(gridcolor='rgba(255,255,255,0.1)'),
        yaxis=dict(gridcolor='rgba(255,255,255,0.1)'),
        zaxis_gridcolor='rgba(255,255,255,0.1)',
    ),
    paper_bgcolor='#0a0a0f',
    font=dict(color='#f8fafc'),
    margin=dict(l=0, r=0, b=0, t=50),
)
fig_surf.show()

## 🎛️ Sandbox — Interactive Controls

Adjust focus threshold, rolling window size, and session number to re-render the time-series chart live.

In [ ]:
output = widgets.Output()

thresh_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.05,
    description='Focus Threshold:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'),
)
window_slider = widgets.IntSlider(
    value=30, min=5, max=100, step=5,
    description='Rolling Window:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'),
)
session_selector = widgets.IntSlider(
    value=1, min=1, max=3, step=1,
    description='Session:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='450px'),
)

def update_ts(focus_threshold, window_size, session):
    with output:
        output.clear_output(wait=True)
        t_s, f_s, _ = sessions[session - 1]
        roll = rolling_avg(f_s, window_size)
        above = np.sum(f_s >= focus_threshold)
        print(f'Session {session} | Window: {window_size} | '
              f'Points above {focus_threshold:.0%}: {above}/{N} ({above/N:.0%})')
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=t_s, y=f_s, mode='lines',
                                 line=dict(color='rgba(99,102,241,0.5)', width=1.5),
                                 name='Focus Score'))
        fig.add_trace(go.Scatter(x=t_s, y=roll, mode='lines',
                                 line=dict(color='#10b981', width=2.5),
                                 name=f'Rolling Avg ({window_size}pt)'))
        fig.add_hline(y=focus_threshold, line_dash='dash',
                      line_color='rgba(245,158,11,0.6)',
                      annotation_text=f'Threshold ({focus_threshold:.0%})')
        fig.update_layout(
            title=f'BCI Focus Score — Session {session}',
            xaxis_title='Time (seconds)', yaxis_title='Focus Score',
            yaxis=dict(range=[0, 1]),
            paper_bgcolor='#0a0a0f', plot_bgcolor='#0a0a0f',
            font=dict(color='#f8fafc'),
            xaxis=dict(gridcolor='rgba(255,255,255,0.07)'),
            yaxis_gridcolor='rgba(255,255,255,0.07)',
        )
        fig.show()

widgets.interact(update_ts, focus_threshold=thresh_slider,
                 window_size=window_slider, session=session_selector)
display(output)